# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# === Feature Engineering básico (como en labs previos) ===
import pandas as pd
import numpy as np

# 1) Limpieza rápida: eliminar filas con NaN (porcentaje bajo de nulos en Spaceship)
df = spaceship.dropna().copy()

# 2) Transformación de 'Cabin' -> 'CabinDeck' (A..G,T) si existe
if 'Cabin' in df.columns:
    df['CabinDeck'] = df['Cabin'].astype(str).str[0]
    valid_decks = list('ABCDEFGT')
    df['CabinDeck'] = df['CabinDeck'].where(df['CabinDeck'].isin(valid_decks))
    df['CabinDeck'] = df['CabinDeck'].astype('category')

# 3) Eliminar identificadores que no aportan al modelo
drop_cols = [c for c in ['PassengerId','Name'] if c in df.columns]
df = df.drop(columns=drop_cols)

# 4) Definir target
target_col = 'Transported'
assert target_col in df.columns, "No se encontró la columna 'Transported'"
y = df[target_col].astype(int)

# 5) Preparar X
X = df.drop(columns=[target_col]).copy()

# Convertir booleanas a 0/1
bool_cols = X.select_dtypes(include=['bool']).columns
for c in bool_cols:
    X[c] = X[c].astype(int)

# One-hot encoding para categóricas
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X.shape, y.shape

((6606, 5323), (6606,))

**Perform Train Test Split**

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado (útil para métodos sensibles a escala; no afecta a árboles, pero lo dejamos preparado)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5284, 5323), (1322, 5323), (5284,), (1322,))

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [5]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

base = DecisionTreeClassifier(random_state=42)

# Bagging (bootstrap=True)
bagging = BaggingClassifier(
    estimator=base, n_estimators=200, max_samples=1.0,
    bootstrap=True, random_state=42, n_jobs=-1
)
bagging.fit(X_train, y_train)  # para árboles no hace falta escalar
y_pred_bag = bagging.predict(X_test)
acc_bag = accuracy_score(y_test, y_pred_bag)
print(f"Bagging (DecisionTree) accuracy: {acc_bag:.3f}")

# Pasting (bootstrap=False)
pasting = BaggingClassifier(
    estimator=base, n_estimators=200, max_samples=1.0,
    bootstrap=False, random_state=42, n_jobs=-1
)
pasting.fit(X_train, y_train)
y_pred_pas = pasting.predict(X_test)
acc_pas = accuracy_score(y_test, y_pred_pas)
print(f"Pasting (DecisionTree) accuracy: {acc_pas:.3f}")


Bagging (DecisionTree) accuracy: 0.782
Pasting (DecisionTree) accuracy: 0.773


- Random Forests

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=400, max_depth=None, min_samples_split=2,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"RandomForest accuracy: {acc_rf:.3f}")

RandomForest accuracy: 0.784


- Gradient Boosting

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)  # árboles débiles; escalado no requerido
y_pred_gb = gb.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)
print(f"GradientBoosting accuracy: {acc_gb:.3f}")

GradientBoosting accuracy: 0.780


- Adaptive Boosting

In [8]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=300, learning_rate=0.5, random_state=42
)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)
acc_ada = accuracy_score(y_test, y_pred_ada)
print(f"AdaBoost accuracy: {acc_ada:.3f}")


AdaBoost accuracy: 0.782


Which model is the best and why?

In [9]:
best = {
    'Bagging': acc_bag if 'acc_bag' in globals() else float('-inf'),
    'Pasting': acc_pas if 'acc_pas' in globals() else float('-inf'),
    'RandomForest': acc_rf if 'acc_rf' in globals() else float('-inf'),
    'GradientBoosting': acc_gb if 'acc_gb' in globals() else float('-inf'),
    'AdaBoost': acc_ada if 'acc_ada' in globals() else float('-inf'),
}
best_model = max(best, key=best.get)
print('Mejor modelo por accuracy:', best_model, f"→ {best[best_model]:.3f}")

comment = f"""
El mejor modelo ha sido {best_model} por accuracy en el conjunto de test.
- En general, los métodos basados en árboles como RandomForest (bagging de árboles) suelen rendir muy bien en datos tabulares con mezcla de variables y sin necesidad de escalado.
- Los métodos de boosting (GradientBoosting/AdaBoost) pueden superar a RF si existen patrones complejos y poco ruido, pero son más sensibles al tuning.
- Para una selección definitiva, conviene validar con cross-validation estratificada y comparar además métricas como F1 y ROC-AUC.
"""
print(comment)

Mejor modelo por accuracy: RandomForest → 0.784

El mejor modelo ha sido RandomForest por accuracy en el conjunto de test.
- En general, los métodos basados en árboles como RandomForest (bagging de árboles) suelen rendir muy bien en datos tabulares con mezcla de variables y sin necesidad de escalado.
- Los métodos de boosting (GradientBoosting/AdaBoost) pueden superar a RF si existen patrones complejos y poco ruido, pero son más sensibles al tuning.
- Para una selección definitiva, conviene validar con cross-validation estratificada y comparar además métricas como F1 y ROC-AUC.

